# 63_rna_de_analysis — RNA Differential Expression: Robust Filtering, Plots & Enrichment

**Goal:** Downstream analysis of RNA pseudobulk DESeq2 results from `61_rna_deseq2.ipynb`:
1. **Robust DE genes** — same `min(pos donors) > max(neg donors)` criterion as ATAC ultra-robust peaks
2. **Volcano plots** per cell type × key contrast (highlighting robust genes)
3. **Species heatmaps** per cell type showing top robust DE genes across primate species
4. **GO enrichment** (Biological Process) on robust UP genes per CT × contrast
5. **Disease enrichment** (KEGG, Disease Ontology, DisGeNET) on robust UP genes
6. **Disease term summary** aggregated across all contrasts

**Outputs (cell-type-first layout under RNA_BASE):**
- `{CT}/rna_robust/{contrast}_robust_DE.csv` — robust UP + DOWN genes
- `{CT}/plots/volcano_{contrast}.pdf` — volcano plots
- `{CT}/plots/heatmap_robust_{contrast}.pdf` — species heatmap
- `{CT}/enrichment/` — GO + disease CSVs and dotplots
- `_summary/RNA_DE_robust_list.rds`
- `_summary/SUMMARY_RNA_disease_terms.csv`

In [1]:
# =============================================================================
# Cell 1: Packages & utilities
# =============================================================================
suppressPackageStartupMessages({
  library(DESeq2)
  library(ggplot2)
  library(ggrepel)
  library(pheatmap)
  library(ComplexHeatmap)
  library(circlize)
  library(viridis)
  library(matrixStats)
  library(clusterProfiler)
  library(DOSE)
  library(org.Hs.eg.db)
  library(enrichplot)
  library(dplyr)
  library(tibble)
  library(readr)
  library(arrow)
})
source("../src/deseq2_utils.R")
message("Packages loaded & utilities sourced.")

Packages loaded & utilities sourced.



In [2]:
# =============================================================================
# Cell 2: Configuration & load RNA DE results
# =============================================================================
BASE_RNA <- "/links/groups/treutlein/USERS/jjans/analysis/adult_intestine/rna"
OUT_DIR  <- file.path(BASE_RNA, "pseudobulk_deseq2")
SPECIES  <- c("Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset")

# Key evolutionary contrasts for deep-dive plots/enrichment
KEY_CONTRASTS <- c(
  "Div_Human_vs_AllPrimates",
  "Div_Human_vs_Apes",
  "Node1_Human_vs_Pan",
  "Node2_AfricanApes_vs_Gorilla",
  "Node3_GreatApes_vs_Macaque",
  "Node4_OldWorld_vs_Marmoset",
  "Node_Apes_vs_Monkeys",
  "ILS_HumanGorilla_vs_Pan",
  "ILS_HumanChimp_vs_GorillaBonobo",
  "ILS_HumanBonobo_vs_ChimpGorilla"
)

# Load evolutionary DE results — try new path first, fall back to old
rds_candidates_evo <- c(
  file.path(OUT_DIR, "_summary", "RNA_DE_res_list.rds"),
  file.path(OUT_DIR, "rna_differential_results", "RNA_DE_res_list.rds")
)
rds_evo <- Filter(file.exists, rds_candidates_evo)[[1]]
rna_res <- readRDS(rds_evo)
message("Loaded evolutionary RNA DE from: ", rds_evo)
message("Cell types: ", length(rna_res))
message("Contrasts per CT (first): ", length(rna_res[[1]]))

# Load species-specific DE results
rds_candidates_sp <- c(
  file.path(OUT_DIR, "_summary", "RNA_DE_species_res_list.rds"),
  file.path(OUT_DIR, "rna_differential_results", "RNA_DE_species_res_list.rds"),
  file.path(OUT_DIR, "rna_differential_results", "species_specific", "RNA_DE_species_res_list.rds")
)
rds_sp_found <- Filter(file.exists, rds_candidates_sp)
if (length(rds_sp_found) > 0) {
  rna_res_species <- readRDS(rds_sp_found[[1]])
  message("Loaded species-specific RNA DE from: ", rds_sp_found[[1]])
} else {
  message("Species-specific RNA DE RDS not found — skipping species enrichment")
  rna_res_species <- NULL
}

# Summary output dir
summary_dir <- file.path(OUT_DIR, "_summary")
dir.create(summary_dir, showWarnings = FALSE, recursive = TRUE)

Loaded evolutionary RNA DE from: /links/groups/treutlein/USERS/jjans/analysis/adult_intestine/rna/pseudobulk_deseq2/rna_differential_results/RNA_DE_res_list.rds



Cell types: 40



Contrasts per CT (first): 19



Loaded species-specific RNA DE from: /links/groups/treutlein/USERS/jjans/analysis/adult_intestine/rna/pseudobulk_deseq2/rna_differential_results/species_specific/RNA_DE_species_res_list.rds



In [3]:
# =============================================================================
# Cell 3: Load pseudobulk counts (for robust DE filter)
# =============================================================================
pb_rds <- file.path(OUT_DIR, "rna_pb_data_aggregated.rds")
pb     <- readRDS(pb_rds)
counts_rna <- pb$counts
meta_rna   <- pb$meta

# Normalize cell_type to match rna_res keys (make.names)
meta_rna$cell_type <- make.names(as.character(meta_rna$cell_type))
meta_rna$species   <- as.character(meta_rna$species)

message("Pseudobulk: ", nrow(counts_rna), " genes x ", ncol(counts_rna), " samples")
message("Samples per species:")
print(table(meta_rna$species))

Pseudobulk: 12785 genes x 672 samples



Samples per species:




    Bonobo Chimpanzee    Gorilla      Human    Macaque   Marmoset 
       103        104        166        151         60         88 


In [4]:
# =============================================================================
# Cell 4: Define contrasts & run robust DE filter
# =============================================================================
evo_contrasts <- define_evolutionary_contrasts()

robust_genes <- ultra_robust_filter_rna(
  rna_res    = rna_res,
  counts_rna = counts_rna,
  meta_rna   = meta_rna,
  contrasts  = evo_contrasts,
  out_dir    = OUT_DIR,
  padj_thresh = 0.05,
  lfc_thresh  = 1,
  min_logcpm  = 0
)

# Summary table: n robust genes per CT × contrast
summary_rows <- list()
for (ct in names(robust_genes)) {
  for (cn in names(robust_genes[[ct]])) {
    n_up <- length(robust_genes[[ct]][[cn]]$up)
    n_dn <- length(robust_genes[[ct]][[cn]]$down)
    summary_rows[[paste0(ct, "__", cn)]] <- data.frame(
      cell_type = ct, contrast = cn,
      n_robust_up = n_up, n_robust_down = n_dn,
      stringsAsFactors = FALSE
    )
  }
}
robust_summary <- do.call(rbind, summary_rows)
write.csv(robust_summary,
          file.path(summary_dir, "RNA_robust_DE_summary.csv"),
          row.names = FALSE)
message("\nRobust summary saved.")
print(robust_summary[robust_summary$contrast == "Div_Human_vs_AllPrimates",
                     c("cell_type","n_robust_up","n_robust_down")])

Defined 19 evolutionary contrasts.




Robust RNA DE filter: TA.cells



  [Node1_Human_vs_Pan]  UP: 1923 sig -> 882 robust  |  DOWN: 1141 sig -> 354 robust



  [Node2_AfricanApes_vs_Gorilla]  UP: 1575 sig -> 231 robust  |  DOWN: 667 sig -> 186 robust



  [Node3_GreatApes_vs_Macaque]  UP: 2014 sig -> 598 robust  |  DOWN: 650 sig -> 448 robust



...


Robust RNA DE filter: Specialized.Fibroblasts.PCDH9.




Robust RNA DE filter: RBP2_high.cells




Robust RNA DE complete. Saved RNA_DE_robust_list.rds




Robust summary saved.



                                                                                        cell_type
TA.cells__Div_Human_vs_AllPrimates                                                       TA.cells
Goblet.cells__Div_Human_vs_AllPrimates                                               Goblet.cells
BEST4..cells__Div_Human_vs_AllPrimates                                               BEST4..cells
Colonocytes__Div_Human_vs_AllPrimates                                                 Colonocytes
Stem.cells__Div_Human_vs_AllPrimates                                                   Stem.cells
EECs__Div_Human_vs_AllPrimates                                                               EECs
T.cells__Div_Human_vs_AllPrimates                                                         T.cells
Lymphatic.ECs__Div_Human_vs_AllPrimates                                             Lymphatic.ECs
Tuft.cells__Div_Human_vs_AllPrimates                                                   Tuft.cells
Specialized.Fibrobla

In [5]:
# =============================================================================
# Cell 5: Volcano plots — per cell type, multi-page PDF (one page per key contrast)
# =============================================================================
for (ct in names(rna_res)) {
  plot_dir <- file.path(OUT_DIR, ct, "plots")
  dir.create(plot_dir, showWarnings = FALSE, recursive = TRUE)
  
  pdf(file.path(plot_dir, "volcano_evolutionary.pdf"), width = 7, height = 5)
  
  for (cn in intersect(KEY_CONTRASTS, names(rna_res[[ct]]))) {
    res_df <- as.data.frame(rna_res[[ct]][[cn]])
    res_df$gene <- rownames(res_df)
    res_df <- res_df[!is.na(res_df$padj), ]
    
    # Classify points
    rob_up <- robust_genes[[ct]][[cn]]$up
    rob_dn <- robust_genes[[ct]][[cn]]$down
    
    res_df$status <- "ns"
    res_df$status[res_df$padj < 0.05 & abs(res_df$log2FoldChange) > 1] <- "sig"
    res_df$status[res_df$gene %in% rob_up] <- "robust_up"
    res_df$status[res_df$gene %in% rob_dn] <- "robust_dn"
    res_df$status <- factor(res_df$status,
                            levels = c("ns", "sig", "robust_up", "robust_dn"))
    
    # Cap -log10 padj for display
    res_df$neglog10p <- pmin(-log10(res_df$padj), 50)
    
    # Top robust genes to label
    top_label <- head(c(rob_up[order(-res_df[rob_up, "log2FoldChange"])],
                        rob_dn[order(res_df[rob_dn,  "log2FoldChange"])]), 20)
    top_label <- top_label[top_label %in% res_df$gene]
    
    p <- ggplot(res_df, aes(log2FoldChange, neglog10p, color = status)) +
      geom_point(data = res_df[res_df$status == "ns", ],
                 size = 0.3, alpha = 0.3, color = "grey70") +
      geom_point(data = res_df[res_df$status == "sig", ],
                 size = 0.5, alpha = 0.5, color = "#6baed6") +
      geom_point(data = res_df[res_df$status == "robust_up", ],
                 size = 1, color = "#d73027") +
      geom_point(data = res_df[res_df$status == "robust_dn", ],
                 size = 1, color = "#4575b4") +
      geom_text_repel(
        data  = res_df[res_df$gene %in% top_label, ],
        aes(label = gene), size = 2.5, max.overlaps = 30,
        color = "black", segment.color = "grey50", segment.size = 0.3
      ) +
      geom_vline(xintercept = c(-1, 1), linetype = "dashed", color = "grey50", linewidth = 0.3) +
      geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "grey50", linewidth = 0.3) +
      scale_color_manual(values = c(ns = "grey70", sig = "#6baed6",
                                    robust_up = "#d73027", robust_dn = "#4575b4"),
                         guide = "none") +
      labs(title = paste(ct, "|", cn),
           subtitle = sprintf("%d robust UP  |  %d robust DOWN",
                              length(rob_up), length(rob_dn)),
           x = "log2 Fold Change", y = "-log10(padj)") +
      theme_bw(base_size = 11)
    
    print(p)
  }
  dev.off()
  message("Volcano PDF: ", ct)
}
message("\nAll volcano plots complete.")

Volcano PDF: TA.cells



Volcano PDF: Goblet.cells



Volcano PDF: BEST4..cells



Volcano PDF: Colonocytes



Volcano PDF: Stem.cells



...

Volcano PDF: Specialized.Fibroblasts.RALYL.



Volcano PDF: Specialized.Fibroblasts.ADAMTS16.



Volcano PDF: Specialized.Fibroblasts.PCDH9.



Volcano PDF: RBP2_high.cells




All volcano plots complete.



In [6]:
# =============================================================================
# Cell 6: Species heatmaps — top robust UP genes per CT for Div_Human_vs_AllPrimates
# =============================================================================
HEATMAP_CONTRAST <- "Div_Human_vs_AllPrimates"
SPECIES_ORDER    <- c("Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque", "Marmoset")
N_TOP            <- 60  # max genes per heatmap

for (ct in names(robust_genes)) {
  rob_up <- robust_genes[[ct]][[HEATMAP_CONTRAST]]$up
  if (length(rob_up) < 5) next

  meta_ct    <- meta_rna[meta_rna$cell_type == ct, ]
  ct_samples <- intersect(rownames(meta_ct), colnames(counts_rna))
  meta_ct    <- meta_ct[ct_samples, ]
  ct_counts  <- counts_rna[, ct_samples, drop = FALSE]

  lib_sizes  <- colSums(ct_counts)
  logcpm_mat <- log2(t(t(ct_counts) / lib_sizes) * 1e6 + 1)

  # Average log2CPM per species
  sp_present <- intersect(SPECIES_ORDER, unique(meta_ct$species))
  avg_mat <- vapply(sp_present, function(sp) {
    samp <- ct_samples[meta_ct$species == sp]
    if (length(samp) > 1) rowMeans(logcpm_mat[, samp, drop = FALSE])
    else                  logcpm_mat[, samp]
  }, numeric(nrow(logcpm_mat)))
  rownames(avg_mat) <- rownames(logcpm_mat)

  # Select top N by LFC, z-score scale
  res_df    <- as.data.frame(rna_res[[ct]][[HEATMAP_CONTRAST]])
  top_genes <- head(rob_up[order(-res_df[rob_up, "log2FoldChange"])], N_TOP)
  mat       <- avg_mat[intersect(top_genes, rownames(avg_mat)), , drop = FALSE]
  if (nrow(mat) < 3) next

  mat_z <- t(apply(mat, 1, scale))
  colnames(mat_z) <- sp_present

  col_fun    <- colorRamp2(c(-2, 0, 2), c("#4575b4", "white", "#d73027"))
  # Scale font so names always fit: 8pt for <=30 genes, scale down to 5pt for 60
  name_size  <- max(5, 8 - (nrow(mat_z) - 30) * 0.1)

  ht <- Heatmap(
    mat_z, name = "Z-score", col = col_fun,
    cluster_columns  = FALSE, cluster_rows = TRUE,
    show_row_names   = TRUE,
    row_names_gp     = gpar(fontsize = name_size),
    row_names_side   = "right",
    column_names_rot = 45,
    column_title     = paste0(ct, "  —  robust UP genes (", HEATMAP_CONTRAST, ")"),
    column_title_gp  = gpar(fontsize = 11, fontface = "bold")
  )

  plot_dir <- file.path(OUT_DIR, ct, "plots")
  dir.create(plot_dir, showWarnings = FALSE, recursive = TRUE)
  h <- max(5, nrow(mat_z) * 0.18 + 2)
  pdf(file.path(plot_dir, paste0("heatmap_robust_", HEATMAP_CONTRAST, ".pdf")),
      width = 7, height = h)
  draw(ht)
  dev.off()
  message("Heatmap saved: ", ct, " (", nrow(mat_z), " genes)")
}
message("\nSpecies heatmaps complete.")

Heatmap saved: TA.cells (60 genes)



Heatmap saved: Goblet.cells (60 genes)



Heatmap saved: BEST4..cells (60 genes)



Heatmap saved: Colonocytes (60 genes)



Heatmap saved: Stem.cells (60 genes)



...

Heatmap saved: Enteric.glia (60 genes)



Heatmap saved: Pericytes (60 genes)



Heatmap saved: ICCs (60 genes)



Heatmap saved: Specialized.Fibroblasts.VCAM1. (60 genes)




Species heatmaps complete.



In [7]:
# =============================================================================
# Cell 7: Cross-CT summary heatmap — n robust UP genes per CT × contrast
# =============================================================================
ct_order <- sort(names(robust_genes))
cn_order <- intersect(KEY_CONTRASTS, unique(unlist(lapply(robust_genes, names))))

n_up_mat <- matrix(0, nrow = length(ct_order), ncol = length(cn_order),
                   dimnames = list(ct_order, cn_order))
n_dn_mat <- n_up_mat
for (ct in ct_order) {
  for (cn in cn_order) {
    if (!is.null(robust_genes[[ct]][[cn]])) {
      n_up_mat[ct, cn] <- length(robust_genes[[ct]][[cn]]$up)
      n_dn_mat[ct, cn] <- length(robust_genes[[ct]][[cn]]$down)
    }
  }
}

col_up <- colorRamp2(c(0, max(1, quantile(n_up_mat, 0.95))),
                     c("white", "#d73027"))
col_dn <- colorRamp2(c(0, max(1, quantile(n_dn_mat, 0.95))),
                     c("white", "#4575b4"))

ht_up <- Heatmap(n_up_mat, name = "Robust UP", col = col_up,
                 cluster_rows = TRUE, cluster_columns = FALSE,
                 row_names_gp = gpar(fontsize = 9),
                 column_names_rot = 45, column_names_gp = gpar(fontsize = 8),
                 column_title = "Robust UP genes",
                 cell_fun = function(j, i, x, y, width, height, fill) {
                   v <- n_up_mat[i, j]
                   if (v > 0) grid.text(v, x, y, gp = gpar(fontsize = 7))
                 })
ht_dn <- Heatmap(n_dn_mat, name = "Robust DOWN", col = col_dn,
                 cluster_rows = TRUE, cluster_columns = FALSE,
                 row_names_gp = gpar(fontsize = 9),
                 column_names_rot = 45, column_names_gp = gpar(fontsize = 8),
                 column_title = "Robust DOWN genes",
                 cell_fun = function(j, i, x, y, width, height, fill) {
                   v <- n_dn_mat[i, j]
                   if (v > 0) grid.text(v, x, y, gp = gpar(fontsize = 7))
                 })

pdf(file.path(summary_dir, "RNA_robust_DE_summary_heatmap.pdf"),
    width = 14, height = max(8, length(ct_order) * 0.4 + 4))
draw(ht_up + ht_dn, merge_legend = TRUE,
     column_title = "RNA Robust DE Genes per Cell Type × Contrast",
     column_title_gp = gpar(fontsize = 13, fontface = "bold"))
dev.off()
message("Summary heatmap saved: ", file.path(summary_dir, "RNA_robust_DE_summary_heatmap.pdf"))

pdf 
  2

Summary heatmap saved: /links/groups/treutlein/USERS/jjans/analysis/adult_intestine/rna/pseudobulk_deseq2/_summary/RNA_robust_DE_summary_heatmap.pdf



In [8]:
# =============================================================================
# Cell 8: GO + disease enrichment on robust UP genes — evolutionary contrasts
#
# NOTE: run_go_enrichment / run_disease_enrichment internally append "enrichment/"
# to out_dir, so pass the CT base dir (not {CT}/enrichment/).
# Skips a CT × contrast if GO output already exists (set FORCE_ENRICH=TRUE to redo).
# =============================================================================
ENRICH_CONTRASTS <- c(
  "Div_Human_vs_AllPrimates", "Div_Human_vs_Apes",
  "Node1_Human_vs_Pan", "Node_Apes_vs_Monkeys",
  "ILS_HumanGorilla_vs_Pan"
)
FORCE_ENRICH <- FALSE

for (ct in names(robust_genes)) {
  ct_dir <- file.path(OUT_DIR, ct)

  for (cn in intersect(ENRICH_CONTRASTS, names(robust_genes[[ct]]))) {
    up_genes <- robust_genes[[ct]][[cn]]$up
    if (length(up_genes) < 5) next

    label      <- paste0(ct, "_", cn)
    go_file    <- file.path(ct_dir, "enrichment", paste0(label, "_GO_BP.csv"))
    if (!FORCE_ENRICH && file.exists(go_file)) {
      message("  [skip] ", ct, " / ", cn, " (enrichment exists)")
      next
    }

    res_df   <- as.data.frame(rna_res[[ct]][[cn]])
    universe <- rownames(res_df[!is.na(res_df$pvalue), ])

    message("\n  [run] ", ct, " / ", cn, " (", length(up_genes), " genes)")
    tryCatch(
      run_go_enrichment(genes = up_genes, label = label,
                        out_dir = ct_dir, universe = universe),
      error = function(e) message("  GO failed: ", e$message)
    )
    tryCatch(
      run_disease_enrichment(genes = up_genes, label = label,
                             out_dir = ct_dir, universe = universe),
      error = function(e) message("  Disease failed: ", e$message)
    )
  }
}
message("\nEvolutionary enrichment complete.")


  [run] TA.cells / Div_Human_vs_AllPrimates (267 genes)



  No significant GO terms for: TA.cells_Div_Human_vs_AllPrimates



'select()' returned 1:1 mapping between keys and columns



  TA.cells_Div_Human_vs_AllPrimates: 267 symbols <e2><86><92> 267 ENTREZ IDs



'select()' returned 1:many mapping between keys and columns



...

    [--] KEGG_Pathways: no significant terms



    [!] DO failed: ontology not supported yet...



    [--] DisGeNET: no significant terms



    [--] Cancer_Genes: no significant terms




Evolutionary enrichment complete.



In [9]:
# =============================================================================
# Cell 9: GO + disease enrichment — species-specific DE genes
# Focus: Human, Marmoset, Macaque. Skips if output already exists.
# =============================================================================
if (!is.null(rna_res_species)) {
  FOCUS_SPECIES <- c("Human", "Marmoset", "Macaque")

  for (ct in names(rna_res_species)) {
    ct_dir <- file.path(OUT_DIR, ct)

    for (sp in intersect(FOCUS_SPECIES, names(rna_res_species[[ct]]))) {
      label   <- paste0(ct, "_", sp, "_vs_rest")
      go_file <- file.path(ct_dir, "enrichment", paste0(label, "_GO_BP.csv"))
      if (!FORCE_ENRICH && file.exists(go_file)) {
        message("  [skip] ", ct, " / ", sp, " vs rest")
        next
      }

      res_df   <- as.data.frame(rna_res_species[[ct]][[sp]])
      universe <- rownames(res_df[!is.na(res_df$pvalue), ])
      up_genes <- rownames(res_df)[!is.na(res_df$padj) &
                                     res_df$padj < 0.05 &
                                     res_df$log2FoldChange > 1]
      if (length(up_genes) < 5) next

      message("\n  [run] ", ct, " / ", sp, " (", length(up_genes), " UP genes)")
      tryCatch(
        run_go_enrichment(genes = up_genes, label = label,
                          out_dir = ct_dir, universe = universe),
        error = function(e) message("  GO failed: ", e$message)
      )
      tryCatch(
        run_disease_enrichment(genes = up_genes, label = label,
                               out_dir = ct_dir, universe = universe),
        error = function(e) message("  Disease failed: ", e$message)
      )
    }
  }
  message("\nSpecies-specific enrichment complete.")
} else {
  message("Skipping species-specific enrichment (RDS not found).")
}


  [run] TA.cells / Human (1140 UP genes)



  No significant GO terms for: TA.cells_Human_vs_rest



'select()' returned 1:1 mapping between keys and columns



Warning message in bitr(genes, fromType = "SYMBOL", toType = "ENTREZID", OrgDb = org.Hs.eg.db, :
"0.35% of input gene IDs are fail to map..."


  TA.cells_Human_vs_rest: 1140 symbols <e2><86><92> 1136 ENTREZ IDs



...

    [OK] KEGG_Pathways: 3 terms



    [!] DO failed: ontology not supported yet...



    [--] DisGeNET: no significant terms



    [--] Cancer_Genes: no significant terms




Species-specific enrichment complete.



In [10]:
# =============================================================================
# Cell 10: Disease term aggregation — collect top terms across all CTs & contrasts
# =============================================================================
disease_pattern <- "_(KEGG|DO|DisGeNET|NCG)_enrichment\\.csv$"
disease_files   <- list.files(OUT_DIR, pattern = disease_pattern,
                               recursive = TRUE, full.names = TRUE)

if (length(disease_files) == 0) {
  message("No disease enrichment files found yet.")
} else {
  disease_rows <- lapply(disease_files, function(f) {
    tryCatch({
      df <- read.csv(f, stringsAsFactors = FALSE)
      if (nrow(df) == 0) return(NULL)
      # Extract CT and contrast from path
      parts <- strsplit(f, .Platform$file.sep)[[1]]
      ct_idx <- which(parts == "enrichment") - 1L
      ct_name <- if (length(ct_idx) > 0) parts[[ct_idx]] else NA_character_
      db_type <- sub(".*_([A-Za-z]+)_enrichment\\.csv$", "\\1", basename(f))
      label   <- sub("_[A-Za-z]+_enrichment\\.csv$", "", basename(f))
      df$source_file <- basename(f)
      df$cell_type   <- ct_name
      df$db          <- db_type
      df$analysis    <- label
      head(df[order(df$p.adjust), ], 10)
    }, error = function(e) NULL)
  })
  
  disease_summary <- do.call(rbind, Filter(Negate(is.null), disease_rows))
  write.csv(disease_summary,
            file.path(summary_dir, "SUMMARY_RNA_disease_terms.csv"),
            row.names = FALSE)
  message("Disease summary saved: ", nrow(disease_summary), " top terms from ",
          length(disease_files), " enrichment files")
  
  # Quick peek at top human-specific disease hits
  human_terms <- disease_summary[
    grepl("Div_Human_vs_AllPrimates", disease_summary$analysis, fixed = TRUE), ]
  if (nrow(human_terms) > 0) {
    message("\nTop disease terms (Div_Human_vs_AllPrimates):")
    print(human_terms[order(human_terms$p.adjust),
                      c("cell_type","db","Description","p.adjust","Count")][1:min(20, nrow(human_terms)), ])
  }
}

No disease enrichment files found yet.



In [11]:
# =============================================================================
# Cell 11: Summary checkpoint
# =============================================================================
message("\n=== 63_rna_de_analysis complete ===")
message("\nRobust DE (Div_Human_vs_AllPrimates):")
for (ct in sort(names(robust_genes))) {
  n_up <- length(robust_genes[[ct]][["Div_Human_vs_AllPrimates"]]$up)
  n_dn <- length(robust_genes[[ct]][["Div_Human_vs_AllPrimates"]]$down)
  if (n_up + n_dn > 0)
    message(sprintf("  %-45s  UP: %4d  DOWN: %4d", ct, n_up, n_dn))
}
message("\nOutputs saved under: ", OUT_DIR, "/{CT}/")
message("  rna_robust/   — robust DE gene tables")
message("  plots/        — volcano PDFs + species heatmaps")
message("  enrichment/   — GO-BP, KEGG, DO, DisGeNET per contrast")
message("\nSummary files: ", summary_dir)
message("  RNA_DE_robust_list.rds")
message("  RNA_robust_DE_summary.csv")
message("  RNA_robust_DE_summary_heatmap.pdf")
message("  SUMMARY_RNA_disease_terms.csv")


=== 63_rna_de_analysis complete ===




Robust DE (Div_Human_vs_AllPrimates):



  Adipocytes                                     UP:   52  DOWN:    4



  BEST4..cells                                   UP:  254  DOWN:   60



  Colonocytes                                    UP:  293  DOWN:  129



...


Summary files: /links/groups/treutlein/USERS/jjans/analysis/adult_intestine/rna/pseudobulk_deseq2/_summary



  RNA_DE_robust_list.rds



  RNA_robust_DE_summary.csv



  RNA_robust_DE_summary_heatmap.pdf



  SUMMARY_RNA_disease_terms.csv

